In [1]:
!pip install polars

In [2]:
!pip install catboost

In [3]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

import gc
import numpy as np
import polars as pl
import json
import pandas as pd
import catboost
from scipy.sparse import csr_matrix, hstack
from catboost import Pool
from catboost import CatBoostClassifier

from sklearn.metrics import roc_auc_score

In [4]:
try:
    import pyarrow
    print("PyArrow is installed. Version:", pyarrow.__version__)
except ModuleNotFoundError:
    print("PyArrow is NOT found in this environment. You installed it in a different one.")

PyArrow is installed. Version: 23.0.1


In [5]:
train = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/train_combined_1600.parquet')

In [6]:
test = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-1600/test_combined_1600.parquet')

In [7]:
target = pl.read_parquet('/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/train_target.parquet')

In [8]:
target_columns = [col for col in target.columns if col.startswith("target")]

In [9]:
cat_feature_names = [
    col_name for col_name in train.columns
    if col_name.startswith("cat_feature")
]

In [10]:
# X_np = train.drop('customer_id').head(50000).to_numpy().astype('float32')
# y_np = (target.drop('customer_id').head(50000).to_numpy().astype('int8'))
# X_test_np = test.drop('customer_id').head(50000).to_numpy().astype('float32')

# X_np = train.drop('customer_id').to_numpy().astype('float32')
# y_np = (target.drop('customer_id').to_numpy().astype('int8'))
#X_test_np = test.drop('customer_id').to_numpy().astype('float32')

In [11]:
# X_main_sparse = csr_matrix(train.drop('customer_id').to_numpy().astype('float32'))
# X_test_sparse = csr_matrix(test.drop('customer_id').to_numpy().astype('float32'))

# del train, test
# gc.collect()

# y_train = target.drop("customer_id").to_pandas()

In [12]:
X_train = train.drop('customer_id').to_pandas().astype('float32')
y_train = target.drop('customer_id').to_pandas()

for col in cat_feature_names:
    if col in X_train.columns:
        # Заполняем NaN каким-то значением (например, -1 или 0)
        X_train[col] = X_train[col].fillna(-1).astype(int)

train_pool = Pool(X_train, label=y_train.values, cat_features = cat_feature_names)

del train
gc.collect()

0

In [13]:
# train_pool = Pool(
#     data=X_main_sparse,
#     label=y_train.values  
# )


catboost_model = CatBoostClassifier(
    iterations=3000,
    depth=8,
    #depth=6,
    learning_rate=0.02,
    random_seed=42,
    loss_function='MultiLogloss',  # <-- для multi-label!
    task_type='GPU',
    #l2_leaf_reg=3.0,
    #l2_leaf_reg=5.0,
    l2_leaf_reg=7.0,
    random_strength=1.5,
    #random_strength=1.0,
)


catboost_model.fit(train_pool)

0:	learn: 0.6454124	total: 2.54s	remaining: 2h 7m 11s
1:	learn: 0.6014641	total: 4.85s	remaining: 2h 1m 6s
2:	learn: 0.5610055	total: 7.46s	remaining: 2h 4m 8s
3:	learn: 0.5237098	total: 10.1s	remaining: 2h 6m 9s
4:	learn: 0.4898218	total: 12.6s	remaining: 2h 6m 4s
5:	learn: 0.4587090	total: 14.9s	remaining: 2h 4m 13s
6:	learn: 0.4302731	total: 17.3s	remaining: 2h 3m 31s
7:	learn: 0.4042697	total: 19.8s	remaining: 2h 3m 13s
8:	learn: 0.3805315	total: 22.2s	remaining: 2h 3m
9:	learn: 0.3588132	total: 24.8s	remaining: 2h 3m 43s
10:	learn: 0.3388968	total: 27.2s	remaining: 2h 3m 11s
11:	learn: 0.3207057	total: 29.8s	remaining: 2h 3m 46s
12:	learn: 0.3040350	total: 32.5s	remaining: 2h 4m 23s
13:	learn: 0.2887447	total: 35.2s	remaining: 2h 5m 9s
14:	learn: 0.2747420	total: 37.9s	remaining: 2h 5m 32s
15:	learn: 0.2618904	total: 40.5s	remaining: 2h 5m 56s
16:	learn: 0.2500003	total: 43.1s	remaining: 2h 5m 56s
17:	learn: 0.2390904	total: 45.8s	remaining: 2h 6m 29s
18:	learn: 0.2291090	total: 4

CatBoostClassifier(depth=8, iterations=3000, l2_leaf_reg=7.0, learning_rate=0.02, loss_function='MultiLogloss', random_seed=42, random_strength=1.5, task_type='GPU')

In [14]:
X_test = test.drop('customer_id').to_pandas().astype('float32')

for col in cat_feature_names:
    if col in X_train.columns:
        # Заполняем NaN каким-то значением (например, -1 или 0)
        X_test[col] = X_test[col].fillna(-1).astype(int)

In [15]:
test_pool = Pool(
    data=X_test,
    cat_features = cat_feature_names
)

y_pred_raw = catboost_model.predict(test_pool, prediction_type='RawFormulaVal')
print(f"Форма сырых значений: {y_pred_raw.shape}")

Форма сырых значений: (250000, 41)


In [16]:
predict_schema = [f"predict_{t.split('target_')[1]}" for t in target_columns]

cat_submit = pl.DataFrame(
    y_pred_raw,
    schema=predict_schema
)

In [17]:
test_ids = pl.read_parquet(
    "/kaggle/input/datasets/mashamalyshkina/kyberpolka-dataset/test_main_features.parquet",
    columns=["customer_id"]
)

In [18]:
cat_submit = pl.concat(
    [test_ids, cat_submit],
    how="horizontal"
)

# Сохраняем
cat_submit.write_parquet("cat_features_1600.parquet")